In [ ]:
from read_model_runs import read_model_runs

# Ler dados completos
question_long_df, model_wide_accuracy_df, question_wide_accuracy_df, firac_order, model_order = read_model_runs('../data/processed/model-runs')

# Filtrar apenas as linhas em português
question_long_df = question_long_df[question_long_df["language"] == "portuguese"].copy()

print('shape', question_long_df.shape)
print('# unique questions', question_long_df['question_id'].nunique())
print('firac_order', firac_order)
print('model_order', model_order)
                                                                                                                                                                        

question_long_df = question_long_df[question_long_df["model_name"].str.contains("gemma", case=False, na=False)]
question_long_df.head(1)



In [ ]:
counts = (
    question_long_df
        .groupby("question_id")
        .size()
        .reset_index(name="row_count")
        .sort_values("row_count", ascending=False)
)

# Total antes do filtro
total_question_ids = len(counts)

# Obtém o count máximo
max_count = counts["row_count"].max()

print('max_count', max_count)

# Filtra apenas os question_id com o count máximo
question_ids = counts.loc[counts["row_count"] == max_count, "question_id"].tolist()

max_count, len(question_ids), total_question_ids, question_ids


max_count 15


(15,
 372,
 537,
 ['oab-1.pdf-003',
  'oab-50.pdf-037',
  'oab-5.pdf-049',
  'oab-5.pdf-048',
  'oab-49.pdf-033',
  'oab-48.pdf-021',
  'oab-48.pdf-017',
  'oab-48.pdf-015',
  'oab-4.pdf-032',
  'oab-4.pdf-031',
  'oab-38.pdf-375',
  'oab-37.pdf-371',
  'oab-37.pdf-367',
  'oab-37.pdf-366',
  'oab-36.pdf-358',
  'oab-36.pdf-356',
  'oab-35.pdf-352',
  'oab-35.pdf-351',
  'oab-35.pdf-348',
  'oab-35.pdf-346',
  'oab-35.pdf-345',
  'oab-35.pdf-344',
  'oab-34.pdf-335',
  'oab-34.pdf-334',
  'oab-34.pdf-333',
  'oab-5.pdf-050',
  'oab-50.pdf-039',
  'oab-178.pdf-266',
  'oab-50.pdf-042',
  'oab-56.pdf-101',
  'oab-56.pdf-100',
  'oab-56.pdf-099',
  'oab-56.pdf-098',
  'oab-56.pdf-096',
  'oab-55.pdf-093',
  'oab-55.pdf-091',
  'oab-55.pdf-089',
  'oab-55.pdf-088',
  'oab-55.pdf-087',
  'oab-55.pdf-085',
  'oab-54.pdf-075',
  'oab-53.pdf-073',
  'oab-53.pdf-068',
  'oab-53.pdf-067',
  'oab-53.pdf-065',
  'oab-52.pdf-062',
  'oab-52.pdf-059',
  'oab-52.pdf-058',
  'oab-52.pdf-056',
  'oab-5

In [ ]:
# antes de restringir...
question_long_df.groupby("materia")["question_id"].nunique()


materia
DIREITO CIVIL             109
DIREITO CONSTITUCIONAL    124
DIREITO TRIBUTÁRIO         76
DIREITOS HUMANOS           37
ÉTICA PROFISSIONAL        191
Name: question_id, dtype: int64

In [ ]:
question_long_df = question_long_df[
    question_long_df["question_id"].isin(question_ids)
]

# depois de restringir...
question_long_df.groupby("materia")["question_id"].nunique()




materia
DIREITO CIVIL              75
DIREITO CONSTITUCIONAL    102
DIREITO TRIBUTÁRIO         54
DIREITOS HUMANOS           18
ÉTICA PROFISSIONAL        123
Name: question_id, dtype: int64

In [ ]:
question_long_df.columns

Index(['model_name', 'firac', 'language', 'is_correct', 'pdf_filename',
       'question_id', 'materia', 'oab_test_id', 'oab_question_id',
       'chosen_option', 'correct_option', 'enunciado', 'prompt',
       'finish_reason', 'avg_logprobs', 'input_token_count',
       'output_token_count', 'A', 'B', 'C', 'D', 'full_response', 'Facts',
       'Issue', 'Rule', 'Application', 'Conclusion', 'rule_count',
       'fact_count', 'response_time_seconds'],
      dtype='object')

In [ ]:
# Calcula a porcentagem de is_correct por question_id e firac
temp = (
    question_long_df
    .groupby(["question_id", "materia", "firac"])
    .agg(perc_correct=("is_correct", "mean"))
    .reset_index()
)

# Converte para uma linha por question_id
question_df = (
    temp
    .pivot(index=["question_id", "materia"], columns="firac", values="perc_correct")
    .reset_index()
)


question_df

firac,question_id,materia,FILA_,FIL__,FIR__,FI___,_____
0,oab-1.pdf-003,ÉTICA PROFISSIONAL,1.000000,0.333333,1.0,0.333333,0.000000
1,oab-1.pdf-004,ÉTICA PROFISSIONAL,1.000000,0.666667,1.0,0.666667,0.333333
2,oab-1.pdf-006,ÉTICA PROFISSIONAL,1.000000,0.666667,1.0,0.666667,1.000000
3,oab-1.pdf-007,ÉTICA PROFISSIONAL,1.000000,0.666667,1.0,1.000000,0.666667
4,oab-1.pdf-008,ÉTICA PROFISSIONAL,0.666667,0.666667,1.0,1.000000,0.333333
...,...,...,...,...,...,...,...
367,oab-9.pdf-085,ÉTICA PROFISSIONAL,1.000000,0.666667,1.0,0.666667,0.666667
368,oab-9.pdf-086,ÉTICA PROFISSIONAL,1.000000,0.666667,1.0,1.000000,1.000000
369,oab-9.pdf-087,ÉTICA PROFISSIONAL,1.000000,0.666667,1.0,0.666667,0.666667
370,oab-9.pdf-089,ÉTICA PROFISSIONAL,1.000000,0.333333,1.0,0.666667,1.000000


In [ ]:
from sklearn.cluster import KMeans
import pandas as pd

# Seleciona apenas colunas numéricas (as porcentagens)
X = question_df.select_dtypes(include=["number"])

K = 3

# Executa K-Means com 3 clusters
kmeans = KMeans(n_clusters=K, random_state=42)
labels = kmeans.fit_predict(X)

# Adiciona os clusters ao question_df
df = question_df.copy()
df["cluster"] = labels

# Centróides encontrados
centroids = kmeans.cluster_centers_

# DataFrame dos centróides (com nomeação dos clusters)
centroids_df = pd.DataFrame(
    centroids,
    columns=X.columns,
    index=[f"cluster_{i}" for i in range(K)]
)

# Número total de questões por matéria
materia_totals = df["materia"].value_counts()

# Número de elementos por cluster
cluster_counts = pd.Series(labels).value_counts().sort_index()
cluster_counts.index = [f"cluster_{i}" for i in range(K)]

# ---- Ordenar clusters por popularidade ----
sorted_clusters = cluster_counts.sort_values(ascending=False)
total_questions = len(df)

print("Clusters ordenados por popularidade:\n")

# Tabela única de centróides (transposta para melhor visualização)
print("="*80)
print("CENTRÓIDES DOS CLUSTERS".center(80))
print("="*80)
print(centroids_df.loc[sorted_clusters.index].round(4))  # Mostra na ordem de popularidade
print("="*80)
print()

for cluster_name in sorted_clusters.index:
    cluster_idx = int(cluster_name.split("_")[1])
    count = sorted_clusters[cluster_name]
    percent_cluster = (count / total_questions) * 100

    print("==============================")
    print(f"{cluster_name} — {count} questões ({percent_cluster:.2f}%)")
    print("==============================")

    # Filtrar questões do cluster
    cluster_subset = df[df["cluster"] == cluster_idx]

    # Contagem por matéria no cluster
    materia_counts = cluster_subset["materia"].value_counts()

    # Porcentagem: (questões da matéria no cluster / total daquela matéria)
    materia_percent = (materia_counts / materia_totals[materia_counts.index]) * 100

    print("\nNúmero de questões por matéria no cluster:")
    print(materia_counts)

    print("\nPorcentagem das questões da matéria que caíram no cluster:")
    print(materia_percent.round(2))

    print("\n")

Clusters ordenados por popularidade:

                            CENTRÓIDES DOS CLUSTERS                             
firac       FILA_   FIL__   FIR__   FI___   _____
cluster_2  0.9786  0.9190  0.9810  0.9381  0.9048
cluster_1  0.9048  0.5514  0.8070  0.6065  0.4712
cluster_0  0.8552  0.1684  0.5993  0.1246  0.1785

cluster_2 — 140 questões (37.63%)

Número de questões por matéria no cluster:
materia
ÉTICA PROFISSIONAL        46
DIREITO CONSTITUCIONAL    43
DIREITO CIVIL             26
DIREITO TRIBUTÁRIO        14
DIREITOS HUMANOS          11
Name: count, dtype: int64

Porcentagem das questões da matéria que caíram no cluster:
materia
ÉTICA PROFISSIONAL        37.40
DIREITO CONSTITUCIONAL    42.16
DIREITO CIVIL             34.67
DIREITO TRIBUTÁRIO        25.93
DIREITOS HUMANOS          61.11
Name: count, dtype: float64


cluster_1 — 133 questões (35.75%)

Número de questões por matéria no cluster:
materia
ÉTICA PROFISSIONAL        38
DIREITO CONSTITUCIONAL    34
DIREITO CIVIL        

In [ ]:
# Adiciona a coluna cluster ao DF original
df = question_df.copy()
df["cluster"] = labels

# Total por matéria
materia_totals = df["materia"].value_counts()

# Cria tabela vazia: linhas = matérias, colunas = clusters
cluster_cols = [f"cluster_{i}" for i in range(K)]
result = pd.DataFrame(index=materia_totals.index, columns=cluster_cols)

# Preencher tabela com porcentagens
for materia in materia_totals.index:
    materia_df = df[df["materia"] == materia]                  # apenas questões da matéria
    counts = materia_df["cluster"].value_counts()              # quantas caíram em cada cluster
    perc = (counts / materia_totals[materia]) * 100            # porcentagem
    
    result.loc[materia, perc.index.map(lambda x: f"cluster_{x}")] = perc.values

# Preenche NaN com 0% (caso matéria não apareça em algum cluster)
result = result.fillna(0).round(2)

print(result)

                        cluster_0  cluster_1  cluster_2
materia                                                
ÉTICA PROFISSIONAL          31.71      30.89      37.40
DIREITO CONSTITUCIONAL      24.51      33.33      42.16
DIREITO CIVIL               22.67      42.67      34.67
DIREITO TRIBUTÁRIO          27.78      46.30      25.93
DIREITOS HUMANOS            16.67      22.22      61.11


C:\Users\pedro\AppData\Local\Temp\ipykernel_13412\2503527976.py:21: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  result = result.fillna(0).round(2)
